In [1]:
"""
Script verifikasi: cocokkan urutan Cluster di CSV kamu
dengan label RFM yang dipakai di dashboard (CLUSTER_META).

Cara pakai:
1. Letakkan file ini di folder yang sama dengan retail_sales_clustered.csv
2. Jalankan: python verify_clusters.py
3. Baca output di terminal -- akan kasih tau apakah CLUSTER_META kamu
   sudah cocok atau perlu ditukar urutannya.
"""

import pandas as pd

df = pd.read_csv("retail_sales_clustered.csv")
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

if 'Cluster' not in df.columns:
    raise SystemExit("Kolom 'Cluster' tidak ditemukan di CSV. Cek nama kolom kamu.")

# Hitung RFM aktual per cluster (sama seperti logika di dashboard.py)
rfm = df.groupby('Cluster').agg(
    Recency   = ('Transaction Date', lambda x: (df['Transaction Date'].max() - x.max()).days),
    Frequency = ('Transaction ID',   'count'),
    Monetary  = ('Total Spent',      'mean'),
    Customers = ('Customer ID',      'nunique'),
).reset_index()

print("="*70)
print("RFM AKTUAL PER CLUSTER (dihitung langsung dari data)")
print("="*70)
print(rfm.to_string(index=False))
print()

# Tentukan cluster mana yang paling cocok untuk tiap label, berdasarkan angka
champions_cluster   = rfm.loc[rfm['Monetary'].idxmax(), 'Cluster']
hibernating_cluster = rfm.loc[rfm['Recency'].idxmax(), 'Cluster']
recent_cluster      = rfm.loc[rfm['Recency'].idxmin(), 'Cluster']

print("="*70)
print("LABEL YANG SEHARUSNYA DIPAKAI (berdasarkan data aktual)")
print("="*70)
print(f"Champions / Top Spenders   -> Cluster {int(champions_cluster)}  (Monetary tertinggi)")
print(f"Hibernating Customers      -> Cluster {int(hibernating_cluster)}  (Recency tertinggi / paling lama tidak belanja)")
print(f"Recent & Active Buyers     -> Cluster {int(recent_cluster)}  (Recency terendah / baru belanja)")
print()

# Bandingkan dengan CLUSTER_META yang dipakai sekarang di dashboard.py
current_mapping = {
    0: "Recent & Active Buyers",
    1: "Hibernating Customers",
    2: "Champions / Top Spenders",
}

correct_mapping = {
    int(recent_cluster):      "Recent & Active Buyers",
    int(hibernating_cluster): "Hibernating Customers",
    int(champions_cluster):   "Champions / Top Spenders",
}

print("="*70)
print("PERBANDINGAN: CLUSTER_META DI KODE KAMU vs YANG SEHARUSNYA")
print("="*70)
mismatch_found = False
for c in sorted(current_mapping.keys()):
    current_label  = current_mapping.get(c, "?")
    correct_label  = correct_mapping.get(c, "?")
    status = "OK" if current_label == correct_label else "!! TIDAK COCOK !!"
    if current_label != correct_label:
        mismatch_found = True
    print(f"Cluster {c}: kode kamu = '{current_label}'  |  seharusnya = '{correct_label}'  ->  {status}")

print()
if mismatch_found:
    print(">> ADA MISMATCH. CLUSTER_META di dashboard.py perlu disesuaikan urutannya.")
    print(">> Gunakan mapping 'correct_mapping' di atas sebagai acuan yang benar.")
else:
    print(">> AMAN. CLUSTER_META di dashboard.py sudah sesuai dengan data aktual.")

RFM AKTUAL PER CLUSTER (dihitung langsung dari data)
 Cluster  Recency  Frequency   Monetary  Customers
       0        0       5966 128.819728         12
       1        2       3427 131.252116          7
       2        0       3182 131.686832          6

LABEL YANG SEHARUSNYA DIPAKAI (berdasarkan data aktual)
Champions / Top Spenders   -> Cluster 2  (Monetary tertinggi)
Hibernating Customers      -> Cluster 1  (Recency tertinggi / paling lama tidak belanja)
Recent & Active Buyers     -> Cluster 0  (Recency terendah / baru belanja)

PERBANDINGAN: CLUSTER_META DI KODE KAMU vs YANG SEHARUSNYA
Cluster 0: kode kamu = 'Recent & Active Buyers'  |  seharusnya = 'Recent & Active Buyers'  ->  OK
Cluster 1: kode kamu = 'Hibernating Customers'  |  seharusnya = 'Hibernating Customers'  ->  OK
Cluster 2: kode kamu = 'Champions / Top Spenders'  |  seharusnya = 'Champions / Top Spenders'  ->  OK

>> AMAN. CLUSTER_META di dashboard.py sudah sesuai dengan data aktual.
